[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SysBioChalmers/MESBcourse/blob/main/exercises/GEM0_precourse_introduction/gem0.ipynb)

# Pre-course exercise: GEMs in cobrapy (solutions)

A Python / cobrapy port of the RAVEN MATLAB exercise.

**Learning goals**

- Run commands in a Python notebook.
- Know what information is contained in a GEM, both in Python objects and in Excel.
- Interpret gene associations (GPR rules): the meaning of `and` and `or`.
- Understand the meaning of a reaction's lower and upper bounds.
- Run flux balance analysis (FBA) and interpret the results.
- Observe with flux variability analysis (FVA) that many alternative optimal flux distributions exist.
- Observe that random sampling also provides knowledge on flux uncertainty.

## Setup

Required packages: `cobra`, `pandas`, `numpy`, `matplotlib`, `openpyxl`.

```bash
pip install cobra pandas numpy matplotlib openpyxl
```

In [ ]:
import sys
!{sys.executable} -m pip install cobra pandas numpy matplotlib openpyxl
!wget -q https://raw.githubusercontent.com/SysBioChalmers/MESBcourse/refs/heads/main/exercises/GEM0_precourse_introduction/yeast-GEM.xml

In [ ]:
import cobra
from cobra.io import read_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from cobra.sampling import sample
from cobra.util import create_stoichiometric_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('cobra version:', cobra.__version__)

## 1. Input and output of models

Genome-scale metabolic models can come in many file formats, but a widely accepted one is SBML (Systems Biology Markup Language), usually with the `.xml` extension. We will use the latest version, level 3, with the Flux Balance Constraint (FBC) package.

SBML is not meant to be human-readable. Many people find it helpful to inspect models in Excel. cobrapy does not export to Excel out of the box, but we can write a small helper that produces a similar layout to RAVEN's `exportToExcelFormat`. The same caveat applies: do not edit the model in Excel, use code instead so changes are tracked.

### 1.1 Import the model

In [ ]:
model_original = read_sbml_model('yeast-GEM.xml')
print(model_original)
print(f'Reactions:   {len(model_original.reactions)}')
print(f'Metabolites: {len(model_original.metabolites)}')
print(f'Genes:       {len(model_original.genes)}')

### 1.2 Export the model to Excel

The helper below builds RXNS, METS, GENES and COMPS sheets, similar to what RAVEN produces.

In [ ]:
def export_to_excel(model, filename):
    """Write a RAVEN-style Excel export of a cobrapy model."""
    rxn_rows = []
    for r in model.reactions:
        subs = r.subsystem
        if isinstance(subs, list):
            subs = '; '.join(subs)
        rxn_rows.append({
            'ID': r.id,
            'NAME': r.name,
            'EQUATION': r.reaction,
            'GENE ASSOCIATION': r.gene_reaction_rule,
            'LOWER BOUND': r.lower_bound,
            'UPPER BOUND': r.upper_bound,
            'OBJECTIVE': r.objective_coefficient,
            'COMPARTMENT': ';'.join(sorted(r.compartments)),
            'SUBSYSTEM': subs or '',
        })
    met_rows = [{
        'ID': m.id, 'NAME': m.name, 'FORMULA': m.formula,
        'CHARGE': m.charge, 'COMPARTMENT': m.compartment,
    } for m in model.metabolites]
    gene_rows = [{'ID': g.id, 'NAME': g.name} for g in model.genes]
    comp_rows = [{'ID': cid, 'NAME': cname} for cid, cname in model.compartments.items()]

    with pd.ExcelWriter(filename) as writer:
        pd.DataFrame(rxn_rows).to_excel(writer, sheet_name='RXNS', index=False)
        pd.DataFrame(met_rows).to_excel(writer, sheet_name='METS', index=False)
        pd.DataFrame(gene_rows).to_excel(writer, sheet_name='GENES', index=False)
        pd.DataFrame(comp_rows).to_excel(writer, sheet_name='COMPS', index=False)

export_to_excel(model_original, 'yeast-GEM.xlsx')
print('Wrote yeast-GEM.xlsx')

Inspect the various sheets in the Excel file to see what information the model carries. You can also inspect a single reaction directly in cobrapy:

In [ ]:
rxn = model_original.reactions.get_by_id('r_2157')
print('ID:           ', rxn.id)
print('Name:         ', rxn.name)
print('Equation:     ', rxn.reaction)
print('GPR:          ', rxn.gene_reaction_rule)
print('Compartments: ', rxn.compartments)
print('Subsystem:    ', rxn.subsystem)
print()
print('Stoichiometry:')
for met, coef in rxn.metabolites.items():
    print(f'  {coef:+g}  {met.id}  ({met.name}) [{met.compartment}]')

### Question 1

Find reaction `r_2157`: in which organelle is this reaction localized?

**Answer.** Endoplasmic reticulum membrane. The compartment code `erm` is clarified in the COMPS sheet of the Excel file (or via `model.compartments` in cobrapy).

In [ ]:
print(model_original.reactions.r_2157.compartments)
print(model_original.compartments)

### Question 2

Can you explain the gene association of reaction `r_2157`? Compare with `r_2142` (a different structural relationship) and `r_0001` (more complex).

In [ ]:
for rid in ['r_2157', 'r_2142', 'r_0001']:
    print(f'{rid}: {model_original.reactions.get_by_id(rid).gene_reaction_rule}')

**Answer.** `"YCR034W or YLR372W"` means two isoenzymes that can each catalyze the reaction independently (logical OR). In `r_2142` the genes are joined by AND, meaning both proteins are required as subunits of one complex. In `r_0001` you see a mix of both (`(A and B) or (C and D)`): two alternative complexes, each requiring two subunits.

### Question 3

Open the SBML file with a text editor and try to find the section where information about reaction `r_2157` is stored. Can you easily fetch the information that is also shown in the RXNS sheet in the Excel file?

**Answer.** No. The SBML block for `r_2157` is verbose XML with separate `<listOfReactants>`, `<listOfProducts>`, `<fbc:geneProductAssociation>`, `<annotation>` and `<notes>` subtrees. Reconstructing a single tabular row from it by eye is painful. This is exactly why we use the Excel export (or cobrapy's accessors) for human-readable inspection.

### Question 4

One of the genes associated to `r_2157` also seems to catalyse a very different reaction. Which reaction is this? Can you find evidence in literature that the protein coded by this gene is involved in that additional catalytic activity?

Hint: in cobrapy you can search for reactions associated with a gene using `model.genes.<gene_id>.reactions`.

In [ ]:
for gid in ['YCR034W', 'YLR372W']:
    g = model_original.genes.get_by_id(gid)
    print(f'{gid} ({g.name}) catalyzes:')
    for r in g.reactions:
        print(f'  {r.id}  {r.name}')
    print()

**Answer.** `YCR034W` (ELO2) also appears in the gene association of `r_0005` (1,3-beta-glucan synthase). That is a very different catalytic activity than an elongase. SGD reports that ELO2 has a regulatory role on 1,3-glucan synthase, and the literature suggests this effect is mediated indirectly via very long fatty acids: ELO2 elongates fatty acids, those very long fatty acids are incorporated in membranes, and 1,3-glucan synthase is a membrane protein. So disrupting ELO2 can affect 1,3-glucan synthase activity, but annotating ELO2 as itself catalyzing (or regulating) 1,3-glucan synthase is misleading. This is a good example of gene associations that come from automated annotation pipelines and need manual curation.

## 2. Model structure

In MATLAB / RAVEN, the model is a struct with parallel arrays (`model.rxns`, `model.lb`, `model.ub`, `model.S`, ...). In cobrapy the model is an object: `model.reactions`, `model.metabolites`, `model.genes` are lists of objects, and the stoichiometric matrix is built on demand. Bounds, GPRs and objective coefficients live on each reaction.

### 2.1 Inspect model content

In [ ]:
print('First 5 reactions:  ', [r.id for r in model_original.reactions[:5]])
print('First 5 metabolites:', [m.id for m in model_original.metabolites[:5]])
print('First 5 genes:      ', [g.id for g in model_original.genes[:5]])
print()

S = create_stoichiometric_matrix(model_original)
print('S shape:', S.shape, '  (rows = metabolites, cols = reactions)')
print()

r = model_original.reactions.r_1714  # D-glucose exchange
print(f'{r.id}  lb={r.lower_bound}  ub={r.upper_bound}  c={r.objective_coefficient}')
print(f'{r.id}  GPR="{r.gene_reaction_rule}"')
print()
print('Current objective:', model_original.objective.expression)

### Question 5

What do the fields `lb`, `ub` and `c` indicate? What values can they take, and what does this indicate?

In cobrapy these are `reaction.lower_bound`, `reaction.upper_bound`, and `reaction.objective_coefficient`.

**Answer.**

- `lb` = lower bound, `ub` = upper bound, `c` = objective coefficient.
- `lb` and `ub` together define the allowed flux range. If `lb = 0`, the reaction is irreversible (no negative flux). If `lb > 0`, the reaction is forced to carry a flux. If `ub < 0`, the reaction can only run in reverse (usually the equation is then defined in the opposite direction). If `lb == ub`, the flux is fixed. Fully unconstrained means `lb = -1000` and `ub = 1000` in this model (RAVEN uses `-Inf` / `+Inf`).
- `c == 1` marks the objective reaction, `c == 0` means it is not the objective. Most reactions have `c == 0`. In cobrapy you can also read the objective with `model.objective` directly.

In [ ]:
obj_rxns = [r for r in model_original.reactions if r.objective_coefficient != 0]
print('Objective reaction(s):', [(r.id, r.objective_coefficient) for r in obj_rxns])

### 2.2 Make changes to the model

### Question 6

If you delete one reaction from the model, which structure fields of the model are changed? (Assume the deleted reaction does not introduce unique metabolites, genes, or compartments.)

**Answer.**

In RAVEN's flat struct: `rxns, lb, ub, S, rev, c, rxnNames, rxnComps, grRules, rxnGeneMat, subSystems, eccodes, rxnMiriams, rxnNotes, rxnReferences, confidenceScores`. All of these contain one entry per reaction.

In cobrapy, all of that information lives on the `Reaction` object itself, so removing the reaction removes everything attached to it in one operation: the entry in `model.reactions`, its column in `S`, its bounds, its objective coefficient, its name, subsystem, annotations, notes, and its connections to genes and metabolites. If the reaction was the only consumer or producer of some metabolite, that metabolite may also be removed (controlled by the `remove_orphans` flag).

In [ ]:
model = model_original.copy()
print('Before:', len(model.reactions), 'reactions')
model.remove_reactions(['r_1237'])
print('After: ', len(model.reactions), 'reactions')
print('Original still has', len(model_original.reactions), 'reactions')

## 3. Flux Balance Analysis

We now assume that the yeast aims to grow as fast as possible: set biomass production as the objective function.

### 3.1 Set parameter values

In [ ]:
model = model_original.copy()
model.objective = 'r_4041'
print(model.reactions.r_4041.reaction)

### Question 7

What are the substrates and products of `r_4041`?

**Answer.** `55.3 ATP[c] + 55.3 H2O[c] + lipid[c] + protein[c] + carbohydrate[c] + RNA[c] + DNA[c] + cofactor[c] + ion[c] => 55.3 ADP[c] + biomass[c] + 55.3 H+[c] + 55.3 phosphate[c]`. Some of these are real metabolites (ATP, H2O, ADP, H+, phosphate), the rest are *pseudometabolites*. For instance, `DNA` is not a real molecule, it represents all the DNA present in a cell, which is produced by its own dedicated reaction.

This model has a nested biomass equation: separate macromolecule reactions feed into `r_4041`.

In [ ]:
rxn_ids = ['r_4041','r_2108','r_4047','r_4048','r_4049',
           'r_4050','r_4063','r_4065','r_4598','r_4599']
for rid in rxn_ids:
    r = model.reactions.get_by_id(rid)
    print(f'{rid}  ({r.name})')
    print(f'    {r.reaction}')

### Question 8

In reaction `r_4041`, what does the use of ATP as substrate represent? And what determines the stoichiometric coefficient?

**Answer.** Hydrolysis of ATP, yielding ADP, phosphate and H+, represents the energy cost of generating biomass. This covers the polymerization of the individual precursors that make up the macromolecules (for example nucleotides linked into DNA, amino acids linked into proteins). The stoichiometric coefficient is how much ATP is required to produce 1 g of biomass.

Limit the glucose uptake rate to 0.5 mmol / gDCW / h:

In [ ]:
model.reactions.r_1714.lower_bound = -0.5
print(model.reactions.r_1714)
print('Equation:', model.reactions.r_1714.reaction)

### Question 9

Pay attention to what we did above. To limit glucose uptake, the *lower* bound is set to `-0.5`. Why not set the *upper* bound to `0.5` instead? Hint: look at the reaction equation.

**Answer.** The reaction is defined as `glucose[e] =>` (from extracellular toward the sink). A positive flux through this reaction means glucose excretion, a negative flux means glucose uptake. So to bound the uptake rate, we constrain the lower bound at `-0.5`.

### 3.2 Inspect flux distributions

In [ ]:
solution = model.optimize()
print('Status:        ', solution.status)
print('Objective (f): ', solution.objective_value)
print()
print(model.summary())

### Question 10

What is the meaning of the objective value in this simulation? Consider what objective function was set.

**Answer.** The objective value is the flux through the objective reaction. We set `r_4041` (biomass production) as objective, so this value is the predicted growth rate (in 1 / h).

To save *all* fluxes (including near-zero ones) to a TSV:

In [ ]:
all_fluxes = pd.DataFrame({
    'rxnID':   [r.id for r in model.reactions],
    'rxnName': [r.name for r in model.reactions],
    'eqn':     [r.reaction for r in model.reactions],
    'flux':    [solution.fluxes[r.id] for r in model.reactions],
})
all_fluxes.to_csv('output.tsv', sep='\t', index=False)
print('Wrote output.tsv with', len(all_fluxes), 'reactions')

print('Flux through r_1237:', solution.fluxes['r_1237'])

## 4. Flux variability analysis

FBA gives one feasible flux distribution at the optimum, but many alternatives exist. FVA gives, for each reaction, the minimum and maximum flux compatible with the optimum.

### Question 11

What growth rate can be reached? In other words, what is the value of the objective? What are the other exchange rates?

**Answer.** Maximum growth rate is approximately `0.0394` / h. Exchange fluxes (from `model.summary()` above) include glucose uptake of `-0.5`, oxygen uptake near `-1.24`, CO2 production near `1.30`, water production near `2.20`, plus small uptake or excretion of phosphate, ammonium, sulfate, and ions. The exact numbers depend on the LP solver but the qualitative picture (aerobic growth on glucose) is the same.

Now fix glucose uptake and growth before running FVA. In RAVEN this is done with `setParam(model, 'var', ..., 5)`, which sets bounds with a 5% variance around a target value. The helper below does the same.

In [ ]:
def set_var(model, rxn_id, value, percent):
    """Set bounds with +/- percent variance around target value (RAVEN-style 'var')."""
    delta = abs(value) * percent / 100.0
    lo, hi = value - delta, value + delta
    model.reactions.get_by_id(rxn_id).bounds = (lo, hi)

model_fva = model_original.copy()
model_fva.objective = 'r_4041'
set_var(model_fva, 'r_1714', -0.5, 5)
sol0 = model_fva.optimize()
print('Max growth at glc=0.5:', sol0.objective_value)
set_var(model_fva, 'r_4041', sol0.objective_value, 5)

In [ ]:
fva = flux_variability_analysis(model_fva, fraction_of_optimum=0.95)
fva.head()

In [ ]:
var_range = (fva['maximum'] - fva['minimum']).sort_values()

frac_zero = (var_range < 1e-10).mean()
frac_unbounded = (var_range > 1999).mean()
print(f'Fraction with variability ~0:    {frac_zero:.2%}')
print(f'Fraction with variability ~2000: {frac_unbounded:.2%}')

plt.figure(figsize=(7, 4))
plt.plot(var_range.values, np.arange(len(var_range)) / len(var_range))
plt.xscale('symlog', linthresh=1e-6)
plt.xlabel('Flux variability range  (max - min)')
plt.ylabel('Cumulative fraction of reactions')
plt.title('CDF of flux variability')
plt.grid(True, alpha=0.3)
plt.show()

### Question 12

Can you think of two reasons why a reaction would have a variability of 0?

**Answer.**

1. The reaction is tightly linked to a constraint we set. Reactions that produce the macromolecule pseudometabolites are directly coupled to `r_4041`, which we fixed before running FVA.
2. The reaction is *blocked*: it can never carry a non-zero flux under the current bounds, so both its min and max are 0.

In [ ]:
highest_var = var_range[var_range > 1999].index.tolist()
print(f'{len(highest_var)} reactions with variability ~2000')
for rid in highest_var[:5]:
    r = model_fva.reactions.get_by_id(rid)
    print(f'  {rid}\t{r.name}')

## 5. Random sampling

FVA returns extreme values; many of these are biologically unrealistic. Random sampling draws flux distributions from the feasible polytope, giving means and standard deviations that are often more informative.

Suppose we cultivated yeast in a bioreactor and measured:

- glucose uptake: 0.5 mmol / gDCW / h  (`r_1714` = -0.5)
- CO2 production: 1.18 mmol / gDCW / h (`r_1672` = 1.18)
- O2 consumption: 1.27 mmol / gDCW / h (`r_1992` = -1.27)
- growth rate:    0.0394 / h           (`r_4041` = 0.0394)

We allow 5% variance on each measurement.

In [ ]:
model_rs = model_original.copy()
model_rs.objective = 'r_4041'
set_var(model_rs, 'r_1714', -0.5,   5)
set_var(model_rs, 'r_1672',  1.18,  5)
set_var(model_rs, 'r_1992', -1.27,  5)
set_var(model_rs, 'r_4041',  0.0394, 5)

for rid in ['r_1714','r_1672','r_1992','r_4041']:
    r = model_rs.reactions.get_by_id(rid)
    print(f'{rid}: bounds = ({r.lower_bound:.4f}, {r.upper_bound:.4f})')

In [ ]:
# cobrapy's default sampler is OptGP. ACHR is also available.
# Note: this differs from RAVEN's sampler, so numerical values will not match
# the original answer key exactly. Qualitative conclusions are the same.
samples = sample(model_rs, n=1000, method='optgp')
print('Samples shape:', samples.shape, '  (rows = samples, columns = reactions)')
samples.iloc[:5, :5]

In [ ]:
samples_clean = samples.where(samples.abs() >= 1e-10, 0.0)

summary = pd.DataFrame({
    'rxnID':   [r.id for r in model_rs.reactions],
    'rxnName': [r.name for r in model_rs.reactions],
    'rxnEq':   [r.reaction for r in model_rs.reactions],
})
summary['mean'] = samples_clean.mean().reindex(summary['rxnID']).values
summary['sd']   = samples_clean.std().reindex(summary['rxnID']).values
summary['min']  = samples_clean.min().reindex(summary['rxnID']).values
summary['max']  = samples_clean.max().reindex(summary['rxnID']).values

summary = summary[['rxnID', 'mean', 'sd', 'min', 'max', 'rxnName', 'rxnEq']]
summary.to_excel('randomSampling.xlsx', index=False)
print('Wrote randomSampling.xlsx')
summary.head()

### Question 13

Inspect the Excel file. Looking at the fluxes, is there anything striking? For the reactions that had high variability after Question 12, what are the mean and standard deviation of their fluxes after random sampling?

In [ ]:
if highest_var:
    sub = summary[summary['rxnID'].isin(highest_var[:10])]
    print(sub[['rxnID', 'mean', 'sd', 'min', 'max', 'rxnName']].to_string(index=False))

**Answer.** A few things may stand out:

1. Many reactions never carry any flux (mean, min and max are all 0). These have a high chance of being *blocked* reactions.
2. Standard deviations are often as large as or larger than the mean flux. This might look like the prediction is poor, but remember that we only fixed a handful of exchange rates, while all other bounds remain at +/- 1000. Increasing the number of samples will lower the standard deviation.
3. Reactions like `r_0280` had an FVA range close to 2000 (effectively unbounded), but in random sampling their mean is not at one extreme. FVA shows what extreme values are *possible*, while random sampling tells you what is *typical* given the constraints. The latter is usually more informative biologically. The min and max in the sampling output also do not reach -1000 and 1000, partly because of the additional constraints we set, and partly because the sampler does not aggressively maximize individual reactions the way FVA does.

## Notes on differences from the RAVEN version

- **SBML inspection (Q3)** is identical: open `yeast-GEM.xml` in a text editor and search for `r_2157`.
- **Model structure (Q5, Q6)**: cobrapy uses objects, not parallel arrays. The information content is the same, just accessed differently. Removing a reaction with `model.remove_reactions(['r_1237'])` updates everything attached to that reaction in one call.
- **Random sampling (Q13)**: cobrapy uses OptGP (or ACHR) rather than RAVEN's sampler, so exact numerical values differ between toolboxes and between runs. The qualitative observations (many reactions carry zero flux, sd often exceeds the mean, the FVA extremes are not representative of typical flux values) all still hold.
- **Solvers**: cobrapy uses optlang and will pick GLPK by default if Gurobi or CPLEX are not installed. To switch: `model.solver = 'gurobi'`.